# 08 SpaceX Falcon 9 - Machine Learning Modeling

This notebook is **Step 08** in a multi-step end-to-end data science project that analyzes and predicts **first-stage landing success** for SpaceX Falcon 9 launches.

In this step we:
- train a few baseline classification models to predict landing success (`Class`)
- tune hyperparameters with cross-validated grid search
- evaluate tuned models on a held-out test split and export a compact performance summary

**Pipeline overview:**

- **Step 01:** Collect launch data from the SpaceX REST API and create an initial modeling dataset.
- **Step 02:** Scrape a *fixed Wikipedia revision* of Falcon 9 & Falcon Heavy launch tables and export a clean CSV for supplementary analysis.
- **Step 03:** Clean and engineer features, create the landing success label (`Class`) and export the modeling dataset.
- **Step 04:** Load the labeled dataset into SQLite and explore patterns with SQL.
- **Step 05:** Visual EDA + one-hot encoding for modeling.
- **Step 06:** Interactive map + proximity distances (Folium).
- **Step 07:** Dashboarding (Dash) for interactive exploration.
- **Step 08 (this notebook):** Train + tune models and compare performance.


**Inputs:**

- `../data/processed/03_dataset_part_2.csv` (labeled launch records including the target `Class`)
- `../data/processed/05_dataset_part_3.csv` (one-hot encoded feature matrix)

**Primary outputs:**

- `../data/processed/08_model_performance.csv` (test accuracy, CV score, best params)
- `../data/processed/08_best_model.json` (selected best model + params)
- `../data/processed/08_best_model.joblib` (serialized fitted pipeline)
- `../data/processed/08_feature_names.json` (feature names expected by the pipeline)

## Notebook sections

1. **Setup**
2. **Load prepared datasets**
3. **Prepare features and target**
4. **Model training + hyperparameter tuning**
5. **Model comparison**
6. **Export artifacts**
7. **Assumptions**

---

## 1. Setup

In [ ]:
from pathlib import Path
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Paths
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

DATASET_PART_2 = PROCESSED_DIR / '03_dataset_part_2.csv'
DATASET_PART_3 = PROCESSED_DIR / '05_dataset_part_3.csv'

# Artifacts
MODEL_PERF_OUT = PROCESSED_DIR / '08_model_performance.csv'
BEST_MODEL_OUT = PROCESSED_DIR / '08_best_model.json'

In [ ]:
# Helper: confusion matrix plot
def plot_confusion_matrix(y, y_predict):
    'This function plots the confusion matrix'
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(y, y_predict)
    ax = plt.subplot()
    sns.heatmap(cm, annot = True, ax = ax); #annot = True to annotate cells
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title('Confusion Matrix'); 
    ax.xaxis.set_ticklabels(['did not land', 'landed']); ax.yaxis.set_ticklabels(['did not land', 'landed']) 
    plt.show()

## 2. Load prepared datasets

We load the labeled dataset from **Step 03** (contains `Class`) and the one-hot encoded feature matrix from **Step 05** (numeric features only).

In [ ]:
# Load engineered launch records
data = pd.read_csv(DATASET_PART_2)

data.head()

In [ ]:
print('dataset_part_2 shape:', data.shape)
print('Class distribution:', data['Class'].value_counts().to_dict())

In [ ]:
# Load one-hot encoded feature matrix
X = pd.read_csv(DATASET_PART_3)

X.head()

In [ ]:
print('dataset_part_3 shape:', X.shape)

## 3. Prepare target and features

- `Y`: target vector (`Class` = 1 successful landing, 0 otherwise)
- `X`: one-hot encoded feature matrix

We standardize features before modeling to keep the setup comparable across model families and to support distance-based methods like kNN and margin-based methods like SVM.

In [ ]:
Y = data['Class'].to_numpy()

In [ ]:
# Split into train/test sets (80/20) using a fixed random seed for reproducibility.
# Note: scaling is applied inside model pipelines during cross-validation (fit per fold).

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size = 0.2,
    random_state = 2,
    stratify = Y
)

In [ ]:
print('Train samples:', Y_train.shape, '| Test samples:', Y_test.shape)

## 4. Model training with hyperparameter search
Each model family is tuned with a small grid search using **10-fold cross validation** on the training split, then evaluated on the held-out test split.

### 4.1 Logistic Regression

Grid search over a small `C` range (L2 penalty).

In [ ]:
# Pipeline: scale within CV folds + logistic regression
logreg_pipe = Pipeline([
    ('scaler', preprocessing.StandardScaler()),
    ('model', LogisticRegression(max_iter = 1000))
])

parameters = {
    'model__C': [0.01, 0.1, 1],
    'model__solver': ['lbfgs'] 
} # penalty l2 is default

logreg_cv = GridSearchCV(logreg_pipe, parameters, cv = 10)
logreg_cv.fit(X_train, Y_train)

In [ ]:
# Best hyperparameters and cross-validated training score (mean CV accuracy)
print('Tuned hyperparameters (best parameters): ', logreg_cv.best_params_)
print('Accuracy: ', logreg_cv.best_score_)

In [ ]:
# Evaluate logistic regression on the held-out test split
logreg_accuracy = logreg_cv.score(X_test, Y_test)
print('Accuracy on test data: ', logreg_accuracy)

In [ ]:
# Confusion matrix for logistic regression
yhat = logreg_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### 4.2 Support Vector Machine (SVC)

In [ ]:
# Pipeline: scale within CV folds + SVC
svm_pipe = Pipeline([
    ('scaler', preprocessing.StandardScaler()),
    ('model', SVC())
])

parameters = {
    'model__kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
    'model__C': np.logspace(-3, 3, 5),
    'model__gamma': np.logspace(-3, 3, 5)
}

In [ ]:
svm_cv = GridSearchCV(svm_pipe, parameters, cv = 10)
svm_cv.fit(X_train, Y_train)

In [ ]:
print('Tuned hyperparameters (best parameters): ', svm_cv.best_params_)
print('Accuracy: ', svm_cv.best_score_)

In [ ]:
# Evaluate SVC on the test split.
svm_accuracy = svm_cv.score(X_test, Y_test)
print('Accuracy on test data: ', svm_accuracy)

In [ ]:
# Confusion matrix for SVC
yhat = svm_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### 4.3 Decision Tree

In [ ]:
# Pipeline: no scaling needed for trees (passthrough)
tree_pipe = Pipeline([
    ('scaler', 'passthrough'),
    ('model', DecisionTreeClassifier(random_state = 2))
])

parameters = {
    'model__criterion': ['gini', 'entropy'],
    'model__splitter': ['best', 'random'],
    'model__max_depth': [2 * n for n in range(1, 10)],
    'model__max_features': ['sqrt', 'log2', None],
    'model__min_samples_leaf': [1, 2, 4],
    'model__min_samples_split': [2, 5, 10]
}

In [ ]:
tree_cv = GridSearchCV(tree_pipe, parameters, cv = 10)
tree_cv.fit(X_train, Y_train)

In [ ]:
print('Tuned hyperparameters (best parameters): ', tree_cv.best_params_)
print('Accuracy: ', tree_cv.best_score_)

In [ ]:
# Evaluate Decision Tree on the test split
tree_accuracy = tree_cv.score(X_test, Y_test)
print('Accuracy on test data: ', tree_accuracy)

In [ ]:
# Confusion matrix for Decision Tree
yhat = tree_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

### 4.4 k-Nearest Neighbors (kNN)

In [ ]:
# Pipeline: scale within CV folds + kNN
knn_pipe = Pipeline([
    ('scaler', preprocessing.StandardScaler()),
    ('model', KNeighborsClassifier())
])

parameters = {
    'model__n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'model__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'model__p': [1, 2]
}

In [ ]:
knn_cv = GridSearchCV(knn_pipe, parameters, cv = 10)
knn_cv.fit(X_train, Y_train)

In [ ]:
print('Tuned hyperparameters (best parameters): ', knn_cv.best_params_)
print('Accuracy: ', knn_cv.best_score_)

In [ ]:
# Evaluate KNN on the test split
knn_accuracy = knn_cv.score(X_test, Y_test)
print('Accuracy on test data: ', knn_accuracy)

In [ ]:
# Confusion matrix for KNN
yhat = knn_cv.predict(X_test)
plot_confusion_matrix(Y_test, yhat)

## 5. Model comparison

We compare test accuracies. If multiple models tie, we break ties using the cross-validated training score from the grid search.

In [ ]:
# Collect results in a single table
results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Support Vector Machine',
        'Decision Tree',
        'K Nearest Neighbors'
    ],
    'Test Accuracy': [
        logreg_accuracy,
        svm_accuracy,
        tree_accuracy,
        knn_accuracy
    ],
    'CV Best Score': [
        logreg_cv.best_score_,
        svm_cv.best_score_,
        tree_cv.best_score_,
        knn_cv.best_score_
    ]
})

# Rank + pick best model (Test Accuracy first, then CV Best Score)
results = results.sort_values(
    by = ['Test Accuracy', 'CV Best Score'],
    ascending = False
).reset_index(drop = True)

best_model = results.iloc[0]['Model']
best_test_accuracy = results.iloc[0]['Test Accuracy']
best_cv = results.iloc[0]['CV Best Score']

# Mark ties on test accuracy
best_test_accuracy_model = results['Test Accuracy'].max()
results['Tied Best Test Accuracy'] = results['Test Accuracy'].eq(best_test_accuracy_model)

display(
    results.style
    .format({'Test Accuracy': '{:.4f}', 'CV Best Score': '{:.4f}'})
    .hide(axis = 'index')
)

print(f'Selected Best Model: {best_model}')
print(f'Test accuracy: {best_test_accuracy:.4f} | CV Best Score: {best_cv:.4f}')

In [ ]:
# Visualize model performance
plot_df = results[['Model', 'Test Accuracy', 'CV Best Score']].copy()
plot_df = plot_df.sort_values('Test Accuracy', ascending = True)
fig, axes = plt.subplots(1, 2, figsize = (12, 4.5), sharey = True)

# Left: Test Accuracy
sns.barplot(data = plot_df, x = 'Test Accuracy', y = 'Model', ax = axes[0])
axes[0].set_title('Test accuracy (hold-out)')
axes[0].set_xlabel('Accuracy')
axes[0].set_ylabel('')

# Right: CV Best Score
sns.barplot(data = plot_df, x = 'CV Best Score', y = 'Model', ax = axes[1])
axes[1].set_title('Best CV score (GridSearchCV)')
axes[1].set_xlabel('Accuracy')
axes[1].set_ylabel('')

# Make the best appear on top (since we sorted ascending)
axes[0].invert_yaxis()

for ax, col in zip(axes, ['Test Accuracy', 'CV Best Score']):
    for p in ax.patches:
        val = p.get_width()
        y = p.get_y() + p.get_height() / 2
        ax.text(val + 0.002, y, f'{val:.3f}', va = 'center')

plt.tight_layout()
plt.show()    

## 6. Export artifacts

To make this final notebook easy to review and reproduce, we export a small set of artifacts that summarize results and enable re-use without re-running the full training pipeline.

**Artifacts written to `../data/processed/`:**
- `08_model_performance.csv`: compact comparison table (test accuracy, best CV score, best params)
- `08_best_model.json`: 'model card' with the selected best model + metadata (split seed, CV folds, library versions)
- `08_best_model.joblib`: serialized fitted pipeline (**StandardScaler + best estimator**) for later inference
- `08_feature_names.json`: ordered feature names expected by the pipeline (from `05_dataset_part_3.csv`)

In [ ]:
# Export performance table + model card (JSON) + serialized best model (joblib)
from datetime import datetime, timezone
import sys

import sklearn
from joblib import dump

# Map model names to their GridSearchCV objects
cv_objects = {
    'Logistic Regression': logreg_cv,
    'Support Vector Machine': svm_cv,
    'Decision Tree': tree_cv,
    'K Nearest Neighbors': knn_cv
}

# Performance summary (CSV)
param_map = {name: cv.best_params_ for name, cv in cv_objects.items()}

model_perf = results[['Model', 'Test Accuracy', 'CV Best Score']].copy()
model_perf['Best Params'] = model_perf['Model'].map(param_map)

# Save a CSV of model performance
model_perf_csv = model_perf.copy()
model_perf_csv['Best Params'] = model_perf_csv['Best Params'].apply(
    lambda d: json.dumps(d, ensure_ascii = False, sort_keys = True)
)
model_perf_csv.to_csv(MODEL_PERF_OUT, index = False)

display(
    model_perf.style
    .format({'Test Accuracy': '{:.4f}', 'CV Best Score': '{:.4f}'})
    .hide(axis = 'index')
)

# Feature names (JSON)
FEATURE_NAMES_OUT = PROCESSED_DIR / '08_feature_names.json'
feature_names = pd.read_csv(DATASET_PART_3, nrows = 0).columns.tolist()
FEATURE_NAMES_OUT.write_text(
    json.dumps(feature_names, indent = 2, ensure_ascii = False),
    encoding = 'utf-8'
)

# Serialized best model (joblib)
MODEL_JOBLIB_OUT = PROCESSED_DIR / '08_best_model.joblib'

best_cv_obj = cv_objects[best_model]
best_estimator = best_cv_obj.best_estimator_

# All GridSearchCV objects are defined on pipelines, so best_estimator is already fitted and self-contained.
dump(
    {'pipeline': best_estimator, 'feature_names': feature_names},
    MODEL_JOBLIB_OUT
)

# Model card (JSON)
best_payload = {
    'selected_best_model': best_model,
    'test_accuracy': float(best_test_accuracy),
    'cv_best_score': float(best_cv),
    'best_params': best_cv_obj.best_params_,
    'pipeline_steps': list(best_estimator.named_steps.keys()) if hasattr(best_estimator, 'named_steps') else None,
    'split': {
        'test_size': 0.2,
        'random_state': 2,
        'stratify': True
    },
    'validation': {
        'cv_folds': 10
    },
    'inputs': {
        'dataset_part_2': DATASET_PART_2.as_posix(),
        'dataset_part_3': DATASET_PART_3.as_posix()
    },
    'outputs': {
        'model_performance_csv': MODEL_PERF_OUT.as_posix(),
        'best_model_json': BEST_MODEL_OUT.as_posix(),
        'best_model_joblib': MODEL_JOBLIB_OUT.as_posix(),
        'feature_names_json': FEATURE_NAMES_OUT.as_posix()
    },
    'environment': {
        'python': sys.version.split()[0],
        'sklearn': sklearn.__version__,
        'numpy': np.__version__,
        'pandas': pd.__version__
    },
    'created_utc': datetime.now(timezone.utc).isoformat()
}

BEST_MODEL_OUT.write_text(
    json.dumps(best_payload, indent = 2, ensure_ascii = False),
    encoding = 'utf-8'
)

## 7. Assumptions
- The target `Class` correctly represents first-stage landing success (1) vs not (0) as engineered in Step 03.
- The feature matrix from Step 05 contains only information available at/before launch; no post-outcome leakage is intentionally included.
- The 80/20 split (random_state = 2) is treated as representative; this is not a time-based split and may overestimate performance if launch conditions drift over time.
- Accuracy is used as the primary metric (equal cost of false positives/false negatives).
- Cross-validation assumes samples are independent (no grouping by launch site, year, or booster category).

---